In [1]:
import jpype
import json
from graphviz import Digraph
from zss import Node, distance
from graphviz import Digraph
from IPython.display import Image, display, SVG
from sysml_utils import SysMLASTManager


parser = SysMLASTManager()


: 

In [ ]:
sysml_code = """
package 'Parts Example-1' {
	part def Vehicle {
		part eng : Engine;
	}
	part def Engine {
		part cyl : Cylinder[4..6];
	}
	part def Cylinder;
	part smallVehicle : Vehicle {
		part redefines eng {
			part redefines cyl[4];
		}
	}
	part bigVehicle : Vehicle {
		part redefines eng {
			part redefines cyl[6];
		}
	}
}
"""


sysml_code_1 = """
package 'Parts Example-1' {
    part def Vehicle {
        part eng : Engine;
    }
    part def Engine {
        part cylinder : Cylinder[4..6];
    }
    part def Cylinder;
    part smallVehicle : Vehicle {
        part redefines eng {
            part redefines cylinder[4];
        }
    }
    part bigVehicle : Vehicle {
        part redefines eng {
            part redefines cylinder[6];
        }
    }
}
"""

sysml_code_2 = """
package 'Parts Example-1' {
    part def Vehicle {
        part eng : Engine;
    }
    part def Engine {
        part cyl : Cylinder[2..8];
    }
    part def Cylinder;
    part smallVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[4];
        }
    }
    part bigVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[6];
        }
    }
}
"""

sysml_code_3 = """
package 'Parts Example-1' {
    part def Vehicle {
        part eng : Engine;
        part wheel : Wheel;
    }
    part def Engine {
        part cyl : Cylinder[4..6];
    }
    part def Cylinder;
    part def Wheel;
    part smallVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[4];
        }
    }
    part bigVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[6];
        }
    }
}
"""

sysml_code_4 = """
package 'Parts Example-1' {
    part def Cylinder;
    part def Vehicle {
        part eng : Engine;
    }
    part def Engine {
        part cyl : Cylinder[4..6];
    }
    part smallVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[4];
        }
    }
    part bigVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[6];
        }
    }
}
"""

sysml_code_5 = """
package 'Parts Example-1' {
    part def Vehicle {
        part eng : Engine;
        part def Engine {
            part cyl : Cylinder[4..6];
        }
    }
    part def Cylinder;
    part smallVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[4];
        }
    }
    part bigVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[6];
        }
    }
}
"""

variations = [sysml_code_1, sysml_code_2, sysml_code_3, sysml_code_4, sysml_code_5]

In [ ]:
ast = parser.code_to_ast(sysml_code)
ast_5 = parser.code_to_ast(sysml_code_5)

simil = parser.compare_asts(ast, ast_5)
print(simil)

In [ ]:
if not jpype.isJVMStarted():
    jpype.startJVM(classpath=[
        ".",                       
        "java_dependencies/*"              
    ])
SysMLParserService = jpype.JClass("SysMLParserService")

In [ ]:
class SysMLASTManager():
    def __init__(self):

        # Use this command in terminal to compile SysMLParserService.java if updated
        # # javac -cp ".;java_dependencies/*" SysMLParserService.java

        # Start the Java Virtual Machine (JVM) and point to the requires .jar and .class files
        if not jpype.isJVMStarted():
            jpype.startJVM(classpath=[".","java_dependencies/*"])

        # Import the parser class. Can take several minutes as it loads all library dependencies
        self.parser = jpype.JClass("SysMLParserService")
    
    def code_to_ast(self, sysml_code):
        ast_json = self.parser.parseSysMLToAST(sysml_code)
        ast = json.loads(ast_json)
        return ast
    
    def _add_nodes_edges(self, graph, node, parent=None):
        node_id = str(id(node))
        label = node.get("type", "?")
        if "name" in node:
            label += f"\n{node['name']}"
        graph.node(node_id, label)
        if parent:
            graph.edge(parent, node_id)
        for child in node.get("children", []):
            self._add_nodes_edges(graph, child, node_id)

    def visualize_ast(self, ast, filename="ast_diagram", display_inline=True, format="png"):
        g = Digraph(comment="SysML AST", format=format)
        self._add_nodes_edges(g, ast)

        if display_inline:
            if format == "svg":
                display(SVG(g.pipe(format="svg")))
            else:
                display(Image(g.pipe(format=format)))
        else:
            output_path = g.render(filename, view=False)
            print(f"Graph saved to {output_path}")
    
    def _dict_to_node(self, d, depth=0):
        n = Node((d.get("type", ""), depth))
        for c in d.get("children", []):
            n.addkid(self._dict_to_node(c, depth + 1))
        return n

    def _insert_cost(self, n):
        _, depth = n.label
        return 1 / (1 + depth)

    def _remove_cost(self, n):
        _, depth = n.label
        return 1 / (1 + depth)

    def _update_cost(self, a, b):
        (atype, adepth), (btype, bdepth) = a.label, b.label

        atype, *aname = atype.split(":", 1)
        btype, *bname = btype.split(":", 1)
        aname = aname[0] if aname else ""
        bname = bname[0] if bname else ""

        if atype != btype:
            # Structural change — big penalty
            return 1.0 / (1 + min(adepth, bdepth))
        elif aname != bname:
            # Rename only — minor penalty
            return 0.05 / (1 + min(adepth, bdepth))
        else:
            return 0.0

    def compare_asts(self, ast1: dict, ast2: dict) -> float:
        tree1 = self._dict_to_node(ast1)
        tree2 = self._dict_to_node(ast2)
        
        dist = distance(
            tree1, 
            tree2,
            get_children=lambda n: n.children,
            insert_cost=self._insert_cost,
            remove_cost=self._remove_cost,
            update_cost=self._update_cost
        )
        
        return 1 / (1 + dist)

In [ ]:
def sysml_to_ast(parser: jpype.JClass, sysml_code: str) -> dict:
    try:
        ast_json = parser.parseString(sysml_code)
        ast = json.loads(str(ast_json))
        return ast
    except jpype.JException as e:
        error = str(e)
        if "ParseException" in error:
            return error 
        else:
            raise

def add_nodes_edges(graph, node, parent=None):
    node_id = str(id(node))
    label = node.get("type", "?")
    if "name" in node:
        label += f"\n{node['name']}"
    graph.node(node_id, label)
    if parent:
        graph.edge(parent, node_id)
    for child in node.get("children", []):
        add_nodes_edges(graph, child, node_id)

def visualize_ast(ast, filename="ast_diagram", display_inline=True, format="png"):
    g = Digraph(comment="SysML AST", format=format)
    add_nodes_edges(g, ast)

    if display_inline:
        # Return or display directly in Jupyter
        if format == "svg":
            display(SVG(g.pipe(format="svg")))
        else:
            display(Image(g.pipe(format=format)))
    else:
        # Save to file
        output_path = g.render(filename, view=False)
        print(f"Graph saved to {output_path}")


In [ ]:
sysml_code = """
package 'Parts Example-1' {
	part def Vehicle {
		part eng : Engine;
	}
	part def Engine {
		part cyl : Cylinder[4..6];
	}
	part def Cylinder;
	part smallVehicle : Vehicle {
		part redefines eng {
			part redefines cyl[4];
		}
	}
	part bigVehicle : Vehicle {
		part redefines eng {
			part redefines cyl[6];
		}
	}
}
"""


sysml_code_1 = """
package 'Parts Example-1' {
    part def Vehicle {
        part eng : Engine;
    }
    part def Engine {
        part cylinder : Cylinder[4..6];
    }
    part def Cylinder;
    part smallVehicle : Vehicle {
        part redefines eng {
            part redefines cylinder[4];
        }
    }
    part bigVehicle : Vehicle {
        part redefines eng {
            part redefines cylinder[6];
        }
    }
}
"""

sysml_code_2 = """
package 'Parts Example-1' {
    part def Vehicle {
        part eng : Engine;
    }
    part def Engine {
        part cyl : Cylinder[2..8];
    }
    part def Cylinder;
    part smallVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[4];
        }
    }
    part bigVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[6];
        }
    }
}
"""

sysml_code_3 = """
package 'Parts Example-1' {
    part def Vehicle {
        part eng : Engine;
        part wheel : Wheel;
    }
    part def Engine {
        part cyl : Cylinder[4..6];
    }
    part def Cylinder;
    part def Wheel;
    part smallVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[4];
        }
    }
    part bigVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[6];
        }
    }
}
"""

sysml_code_4 = """
package 'Parts Example-1' {
    part def Cylinder;
    part def Vehicle {
        part eng : Engine;
    }
    part def Engine {
        part cyl : Cylinder[4..6];
    }
    part smallVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[4];
        }
    }
    part bigVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[6];
        }
    }
}
"""

sysml_code_5 = """
package 'Parts Example-1' {
    part def Vehicle {
        part eng : Engine;
        part def Engine {
            part cyl : Cylinder[4..6];
        }
    }
    part def Cylinder;
    part smallVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[4];
        }
    }
    part bigVehicle : Vehicle {
        part redefines eng {
            part redefines cyl[6];
        }
    }
}
"""

variations = [sysml_code_1, sysml_code_2, sysml_code_3, sysml_code_4, sysml_code_5]

In [ ]:
# Parse the SysML code and get the AST in JSON format. Very quick.
#ast_json = SysMLParserService.parseString(sysml_code)
#ast_json = SysMLParserService.parseFile("test2.sysml")
#ast = json.loads(str(ast_json))
#visualize_ast(ast)
#print(json.dumps(ast, indent=2))
ast = sysml_to_ast(sysml_code)
ast_1 = sysml_to_ast(sysml_code_1)
#print(json.dumps(ast_1, indent=2))

In [ ]:
ast = sysml_to_ast(SysMLParserService, sysml_code)
print(json.dumps(ast_1, indent=2))

In [ ]:
status = jpype.isJVMStarted()
print(status)
jpype.shutdownJVM()

In [ ]:
def dict_to_node(d, depth=0):
    n = Node((d.get("type", ""), depth))
    for c in d.get("children", []):
        n.addkid(dict_to_node(c, depth + 1))
    return n

def insert_cost(n):
    _, depth = n.label
    return 1 / (1 + depth)

def remove_cost(n):
    _, depth = n.label
    return 1 / (1 + depth)

def update_cost(a, b):
    (atype, adepth), (btype, bdepth) = a.label, b.label

    # Split out type:name if needed
    atype, *aname = atype.split(":", 1)
    btype, *bname = btype.split(":", 1)
    aname = aname[0] if aname else ""
    bname = bname[0] if bname else ""

    if atype != btype:
        # Structural change — big penalty
        return 1.0 / (1 + min(adepth, bdepth))
    elif aname != bname:
        # Rename only — minor penalty
        return 1 / (1 + min(adepth, bdepth))
    else:
        return 0.0

def compare_asts(ast1: dict, ast2: dict) -> float:
    tree1 = dict_to_node(ast1)
    tree2 = dict_to_node(ast2)
    
    dist = distance(
        tree1, 
        tree2,
        get_children=lambda n: n.children,
        insert_cost=insert_cost,
        remove_cost=remove_cost,
        update_cost=update_cost
    )
    
    return 1 / (1 + dist)

In [ ]:
ast = sysml_to_ast(sysml_code)
for variation in variations:
    ast_var = sysml_to_ast(variation)
    similarity = compare_asts(ast, ast_var)
    print(f"AST Similarity: {similarity:.4f}")

#visualize_ast(ast, filename="ast_diagram")
#visualize_ast(ast_1, filename="ast_diagram_1")